# Combine VLM sources globally

* Cities (Shirzei et al., 2024)
* Deltas (Ohenhen et al., 2026)
* Europe (EGMS)
* China (Ao et al., 2024)
* US (Ohenhen et al., 2024)
* New Zealand (Hamling et al., 2022)

Approaches:
* Average
* Median

In [1]:
from settings_and_functions import *

# Combine and merge all datasets to DIVA model grid

In [2]:
### LOAD ALL DATASETS

# VLM reconstruction and point-wise trends
VLM_map  = xr.open_dataset(settings['data_in']+'/OE24/VLM_reconstruction.nc')
points_trends_med = xr.open_dataset(settings['data_in']+'/OE24/Vertical_land_motion_point_wise_trends.nc')

# GIA
GIA = xr.open_dataset(settings['data_in']+'Caron2018/GIA_maps_Caron_et_al_2018.nc')
GIA['trend'].attrs['long_name']='linear trend (GIA, C18)'

# InSAR
InSAR_eu = divide_vals(xr.open_dataset(settings['data_in']+'InSAR_EU/EGMS_DIVA.nc'))
InSAR_us = divide_vals(xr.open_dataset(settings['data_in']+'InSAR_US/US_DIVA.nc'))
InSAR_china = divide_vals(xr.open_dataset(settings['data_in']+'/InSAR_China/China_DIVA.nc'))
InSAR_cities = divide_vals(xr.open_dataset(settings['data_in']+'/InSAR_Cities/InSAR_Cities_DIVA.nc'))
InSAR_NZ = divide_vals(xr.open_dataset(settings['data_in']+'/InSAR_NZ/NZ_DIVA.nc'))


# DIVA grid ()
dat_info = pd.read_csv(settings['data_in']+'Nicholls2021/41558_2021_993_MOESM3_ESM.csv')
ds = xr.Dataset({'name': (['x'],  dat_info['locationid']),'test_val': (['x'],  [0] * len(dat_info['locationid']))},
                coords={'lon': (['x'], dat_info['long']),
                'lat': (['x'], dat_info['lat'])}) 

dat_info_gr= dat_info.drop_duplicates(subset = 'locationid')
ds_coords = xr.Dataset({'name': (['x'],  dat_info_gr['locationid']),'test_val': (['x'],  [0] * len(dat_info_gr['locationid']))},
                coords={'lon': (['x'], dat_info_gr['long']),
                'lat': (['x'], dat_info_gr['lat'])}) 

# cmems
cmems_trend = xr.open_dataset(settings['data_in']+'/CMEMS/CMEMS_sla_mm_trend_monthly_1995_2019.nc')

# Map datasets on DIVA HR

In [ ]:
# map VLM map on DIVA

data_comb = []
# artificial way to increase size of first array (needed because sl().couple always maps the larger onto the smaller array, and DIVA should be the target array)
for i in range(14):
    data_comb.append(VLM_map['VLM_trend_coefficient_mean'])

data_out = xr.concat(data_comb,dim='x') 
map_1_point,map_2_point = sl(data_out).couple(ds['test_val'],limit=100)
ds['VLM_map']=map_1_point.data

data_comb = []

for i in range(14):
    data_comb.append(VLM_map['VLM_trend_coefficient_uncertainty'])
data_out = xr.concat(data_comb,dim='x')    
map_1_point,map_2_point = sl(data_out).couple(ds['test_val'],limit=100)
ds['VLM_map_uncertainty']=map_1_point.data

nearest
test_val
100


# Map on DIVA LR 
ds_grouped: DIVA grid with mean statistics of GIA, VLM map (OE24), and CMEMS

In [ ]:
# Note, that lengthscales of the datasets in this cell are much larger the the DIVA LR gridcell size, so mean is appropriate here

ds_grouped = ds.groupby('name').mean()



# add population
data_info_2 = pd.read_csv(settings['data_in']+'Nicholls2021/41558_2021_993_MOESM2_ESM.csv')
data_lower = data_info_2[(data_info_2['subsidence'] == 'lower estimate') & (data_info_2['variable'] == 'Population-weighted coastal relative sea level')]
data_upper = data_info_2[(data_info_2['subsidence'] == 'upper estimate') & (data_info_2['variable'] == 'Population-weighted coastal relative sea level')]
pop_ = []
for name in ds_grouped['name'].values:
    val_= data_lower[data_lower['locationid']==name]['pop_below_10p0'].values
    pop_.append(val_[0])
    
length = []
for name in ds_grouped['name'].values:
    val_= data_lower[data_lower['locationid']==name]['length'].values
    length.append(val_[0])
ds_grouped['length'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
ds_grouped['length'][:] = length

ds_grouped['pop_below_10p0'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
ds_grouped['GIA'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
ds_grouped['GIA_un'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
ds_grouped['ASL'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
ds_grouped['ASL_uncertainty'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
ds_grouped['pop_below_10p0'][:] = pop_
ds_grouped = ds_grouped.sortby('name')
ds_grouped = ds_grouped.assign_coords({"lon":('name',ds_coords.sortby('name').lon.values ),'lat':('name',ds_coords.sortby('name').lat.values)})

# GIA
map_1_point,map_2_point = sl(GIA['trend']).couple(ds_grouped['test_val'],limit=100)
ds_grouped['GIA'][:] = map_1_point.data
map_1_point,map_2_point = sl(GIA['trend_un']).couple(ds_grouped['test_val'],limit=100)
ds_grouped['GIA_un'][:] = map_1_point.data    

# ASL
cmems_trend_un_flat = flatten_lon_lat(cmems_trend['trend_un'],name='trend_un',time=False)
cmems_trend_lat = flatten_lon_lat(cmems_trend['trend'],name='trend',time=False)
# Couple ALS and DIVA grid
map_1_point,map_2_point = sl(cmems_trend_un_flat['trend_un'][np.isfinite(cmems_trend_un_flat['trend_un'])]).couple(ds_grouped['test_val'],limit=500)
ds_grouped['ASL_uncertainty'][:] = map_1_point.data    
map_1_point,map_2_point = sl(cmems_trend_lat['trend'][np.isfinite(cmems_trend_lat['trend'])]).couple(ds_grouped['test_val'],limit=500)
ds_grouped['ASL'][:] = map_1_point.data    

ds_grouped = ds_grouped.rename_dims({'name':'x'})

ds_grouped['VLM_map_ori'] = copy.deepcopy(ds_grouped['VLM_map'])
# Mask some regions with poor GNSS station coverage (will later be filled with GIA, single station GPS and InSAR)
mask =((ds_grouped.lon > 55.36) & (ds_grouped.lon < 73.55) & (ds_grouped.lat > 13.51) &(ds_grouped.lat <29.67 ))

ds_grouped['VLM_map'][mask]=np.nan
ds_grouped['VLM_map_uncertainty'][mask]=np.nan


nearest
test_val
100
nearest
test_val
100


NameError: name 'isfinite' is not defined

# Add more information from Nicholls et al., 2021

In [10]:
# Merge VLM_map with Nicholls delta and city subsidence only  

data = pd.read_csv(settings['data_in']+'Nicholls2021/41558_2021_993_MOESM6_ESM.csv',sep=',')
variables = data['variable'].drop_duplicates().values
for var in variables:
    data_sub = data[data['variable'] ==var].sort_values(by='locationid')
    ds_grouped[var] = copy.deepcopy(ds_grouped['test_val'])*0
    ds_grouped[var][np.isin(ds_grouped['name'],data_sub['locationid'])] =data_sub['value'][np.isin(data_sub['locationid'],ds_grouped['name'])]/1000.
    
# Nicholls delta and city subsidence only    
ds_grouped['VLM_map_delta_city'] = copy.deepcopy(ds_grouped['VLM_map'])
non_zero = ~((ds_grouped['(c) Delta subsidence only']==0) & (ds_grouped['(d) City subsidence only']==0)).values
ds_grouped['VLM_map_delta_city'][non_zero]=0
ds_grouped['VLM_map_delta_city'][non_zero] =- ds_grouped['(c) Delta subsidence only'][non_zero] - ds_grouped['(d) City subsidence only'][non_zero]

# Compute RSL and add GIA to VLM_map at gaps

In [11]:
ds_grouped['RSL_map_delta_city'] = ds_grouped['ASL'] - ds_grouped['VLM_map_delta_city']

for name in ['VLM_map','VLM_map_delta_city']:
    ds_grouped[name+'_GIA'] = copy.deepcopy(ds_grouped[name])
    ds_grouped[name+'_GIA'][np.isnan(ds_grouped[name+'_GIA'])] = ds_grouped['GIA'][np.isnan(ds_grouped[name+'_GIA'])]/1000.
    ds_grouped['RSL_'+name+'_GIA'] = ds_grouped['ASL'] - ds_grouped[name+'_GIA']
    
for name in ['VLM_map_uncertainty']:
    ds_grouped[name+'_GIA'] = copy.deepcopy(ds_grouped[name])
    ds_grouped[name+'_GIA'][np.isnan(ds_grouped[name+'_GIA'])] = ds_grouped['GIA_un'][np.isnan(ds_grouped[name+'_GIA'])]/1000.
ds_grouped['ASL_uncertainty'][:] = ds_grouped['ASL_uncertainty'].to_dataframe()['ASL_uncertainty'].replace(np.inf, np.nan)
ds_grouped['ASL_plus_GIA'] = ds_grouped['ASL'] - ds_grouped['GIA']/1000.

for name in ['VLM_map_uncertainty','VLM_map_uncertainty_GIA']:
    ds_grouped['combined_unc_'+name] = ds_grouped['ASL_uncertainty']/2 + ds_grouped[name]      # ASL uncertainty ~ 2 STDEVS (VLM OE24 and GIA ~ 1 STDEV)


# Combine datasets MEAN and MEDIAN

In [19]:
ds_grouped['datatype'] = copy.deepcopy(ds_grouped['VLM_map']*np.nan)

for method in ['mean','median']:
    # FILL GRID Iteratively with data
    for option,names in zip([2,3,3.01,3.02,3.03],['Cities','China','US','EU','US_EGMS']): 

        data_name = 'InSAR'+str(option)
        if method =='median':
            data_name = 'InSAR'+str(option)+'_'+method

        InSAR_cities_grouped = getattr(InSAR_cities.groupby("locationid"), method)()
        InSAR_cities_grouped_std = InSAR_cities.groupby("locationid").std()

        if method =='mean':
            # uncertainty weighted average for trends
            InSAR_cities_grouped['trend'][:] = InSAR_cities.to_dataframe().groupby('locationid').apply(lambda x: np.average(x['trend'],weights = 1/x['trend_un'])).values

        # Cities
        ds_grouped['VLM_'+data_name] = copy.deepcopy(ds_grouped['VLM_map']*np.nan)
        ds_grouped['VLM_'+data_name+'_uncertainty'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)
        ds_grouped['VLM_'+data_name+'_rob_uncertainty'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty']*np.nan)

        ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_cities_grouped['locationid'].values)] =InSAR_cities_grouped['trend'][np.isin(InSAR_cities_grouped['locationid'].values,ds_grouped.name.values)]
        ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_cities_grouped['locationid'].values)] =InSAR_cities_grouped['trend_un'][np.isin(InSAR_cities_grouped['locationid'].values,ds_grouped.name.values)]

        un1 = InSAR_cities_grouped['trend_un'][np.isin(InSAR_cities_grouped['locationid'].values,ds_grouped.name.values)]
        un2 = InSAR_cities_grouped_std['trend'][np.isin(InSAR_cities_grouped['locationid'].values,ds_grouped.name.values)]


        ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_cities_grouped_std['locationid'].values)] =np.nanmax([un2.values,un1.values],axis=0)
        
        ds_grouped['datatype'][np.isin(ds_grouped.name.values,InSAR_cities_grouped['locationid'].values)]=0

        # New Zealand
        InSAR_NZ_grouped = getattr(InSAR_NZ.groupby("name"), method)()
        InSAR_NZ_grouped_std = InSAR_NZ.groupby("name").std()

        ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_NZ_grouped['name'].values)] =InSAR_NZ_grouped['trend'][np.isin(InSAR_NZ_grouped['name'].values,ds_grouped.name.values)].values
        ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_NZ_grouped['name'].values)] =InSAR_NZ_grouped['trend_un'][np.isin(InSAR_NZ_grouped['name'].values,ds_grouped.name.values)].values

        un1 = InSAR_NZ_grouped['trend_un'][np.isin(InSAR_NZ_grouped['name'].values,ds_grouped.name.values)]
        un2 = InSAR_NZ_grouped_std['trend'][np.isin(InSAR_NZ_grouped['name'].values,ds_grouped.name.values)]

        ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_NZ_grouped['name'].values)] =np.nanmax([un2.values,un1.values],axis=0)

        ds_grouped['datatype'][np.isin(ds_grouped.name.values,InSAR_NZ_grouped['name'].values)]=1

        if option>2:
            # China
            InSAR_china_grouped_un = InSAR_china.groupby("locationid").median() # Spatial trend uncertainties (based on prob. distirbutions per city)
            InSAR_china_grouped= getattr(InSAR_china.groupby("locationid"), method)()
            # Only use values where InSAR_cities_grouped is not defined
            InSAR_china_grouped_un = InSAR_china_grouped_un.isel({'locationid':~np.isin(InSAR_china_grouped.locationid,InSAR_cities_grouped.locationid)})
            InSAR_china_grouped = InSAR_china_grouped.isel({'locationid':~np.isin(InSAR_china_grouped.locationid,InSAR_cities_grouped.locationid)})


            ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_china_grouped['locationid'].values)] =InSAR_china_grouped['trend'][np.isin(InSAR_china_grouped['locationid'].values,ds_grouped.name.values)]
            ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_china_grouped['locationid'].values)] =InSAR_china_grouped['trend_un'][np.isin(InSAR_china_grouped['locationid'].values,ds_grouped.name.values)]*0 # we only use spatial but no formal uncertainties here (they are not provided by Ao et al., 2024)

            un1 = InSAR_china_grouped_un['trend_un'][np.isin(InSAR_china_grouped['locationid'].values,ds_grouped.name.values)]

            ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_china_grouped['locationid'].values)] =un1.values
            ds_grouped['datatype'][np.isin(ds_grouped.name.values,InSAR_china_grouped['locationid'].values)]=2    

        if option ==3.01:
            # US
            InSAR_us_grouped = getattr(InSAR_us.groupby("locationid"), method)() 
            InSAR_us_grouped_std = InSAR_us.groupby("locationid").std()

            ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)] =InSAR_us_grouped['trend'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]
            ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)] =InSAR_us_grouped['trend_un'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]

            un1 = InSAR_us_grouped['trend_un'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]
            un2 = InSAR_us_grouped_std['trend'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]

            ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)] =np.nanmax([un2.values,un1.values],axis=0)
                        
            ds_grouped['datatype'][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)]=3
        if option ==3.02:
            # EU
            InSAR_eu_grouped= getattr(InSAR_eu.groupby("locationid"), method)()
            InSAR_eu_grouped_std = InSAR_eu.groupby("locationid").std()

            InSAR_eu_grouped['trend'][:] = (getattr(InSAR_eu['trend'].groupby('locationid'), method)()+v_lat(InSAR_eu['lat'].groupby('locationid').mean().values)/1000.).values
            ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)] =InSAR_eu_grouped['trend'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]
            ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)] =InSAR_eu_grouped['trend_un'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]

            un1 = InSAR_eu_grouped['trend_un'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]
            un2 = InSAR_eu_grouped_std['trend'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]

            ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)] =np.nanmax([un2.values,un1.values],axis=0)
            ds_grouped['datatype'][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)]=4
        if option >3.02:
            # EU and US
            InSAR_us_grouped = getattr(InSAR_us.groupby("locationid"), method)() 
            InSAR_us_grouped_std = InSAR_us.groupby("locationid").std()

            ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)] =InSAR_us_grouped['trend'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]
            ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)] =InSAR_us_grouped['trend_un'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]
            
            un1 = InSAR_us_grouped['trend_un'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]
            un2 = InSAR_us_grouped_std['trend'][np.isin(InSAR_us_grouped['locationid'].values,ds_grouped.name.values)]

            
            ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_us_grouped['locationid'].values)] =np.nanmax([un2.values,un1.values],axis=0)
                        
            InSAR_eu_grouped= getattr(InSAR_eu.groupby("locationid"), method)()
            InSAR_eu_grouped_std = InSAR_eu.groupby("locationid").std()

            InSAR_eu_grouped['trend'][:] = (getattr(InSAR_eu['trend'].groupby('locationid'), method)()+v_lat(InSAR_eu['lat'].groupby('locationid').mean().values)/1000.).values
            ds_grouped['VLM_'+data_name][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)] =InSAR_eu_grouped['trend'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]
            ds_grouped['VLM_'+data_name+'_uncertainty'][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)] =InSAR_eu_grouped['trend_un'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]

            un1 = InSAR_eu_grouped['trend_un'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]
            un2 = InSAR_eu_grouped_std['trend'][np.isin(InSAR_eu_grouped['locationid'].values,ds_grouped.name.values)]


            ds_grouped['VLM_'+data_name+'_rob_uncertainty'][np.isin(ds_grouped.name.values,InSAR_eu_grouped['locationid'].values)] =np.nanmax([un2.values,un1.values],axis=0)
                        
            # -> until here: China, Cities, US, and EGMS

        # Add InSAR to VLM_map wherever available
        ds_grouped['VLM_map_'+data_name] = copy.deepcopy(ds_grouped['VLM_map'])
        ds_grouped['VLM_map_'+data_name+'_uncertainty'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty'])
        ds_grouped['VLM_map_'+data_name+'_rob_uncertainty'] = copy.deepcopy(ds_grouped['VLM_map_uncertainty'])

        ds_grouped['VLM_map_'+data_name][~np.isnan(ds_grouped['VLM_'+data_name])] = ds_grouped['VLM_'+data_name].dropna(dim='x').values
        ds_grouped['VLM_map_'+data_name+'_uncertainty'][~np.isnan(ds_grouped['VLM_'+data_name+'_uncertainty'])] = ds_grouped['VLM_'+data_name+'_uncertainty'].dropna(dim='x').values
        ds_grouped['VLM_map_'+data_name+'_rob_uncertainty'][~np.isnan(ds_grouped['VLM_'+data_name+'_rob_uncertainty'])] = ds_grouped['VLM_'+data_name+'_rob_uncertainty'].dropna(dim='x').values

        ds_grouped['RSL_map_'+data_name] = ds_grouped['ASL'] - ds_grouped['VLM_map_'+data_name]


        for name in ['VLM_map_'+data_name]:
            ds_grouped[name+'_GIA'] = copy.deepcopy(ds_grouped[name])
            ds_grouped[name+'_GIA'][np.isnan(ds_grouped[name+'_GIA'])] = ds_grouped['GIA'][np.isnan(ds_grouped[name+'_GIA'])]/1000.
            ds_grouped['RSL_'+name+'_GIA'] = ds_grouped['ASL'] - ds_grouped[name+'_GIA']

        for name in ['VLM_map_'+data_name+'_uncertainty','VLM_map_'+data_name+'_rob_uncertainty']:
            ds_grouped[name+'_GIA'] = copy.deepcopy(ds_grouped[name])
            ds_grouped[name+'_GIA'][np.isnan(ds_grouped[name+'_GIA'])] = ds_grouped['GIA_un'][np.isnan(ds_grouped[name+'_GIA'])]/1000.

        ds_grouped['ASL_uncertainty'][:] = ds_grouped['ASL_uncertainty'].to_dataframe()['ASL_uncertainty'].replace(np.inf, np.nan)
        ds_grouped['ASL_plus_GIA'] = ds_grouped['ASL'] - ds_grouped['GIA']/1000.
        ds_grouped['VLM_'+data_name+'_nonzero'] = copy.deepcopy(ds_grouped['VLM_'+data_name])
        ds_grouped['VLM_'+data_name+'_nonzero'][np.isnan(ds_grouped['VLM_'+data_name])] =0
        for name in ['VLM_map_'+data_name+'_uncertainty','VLM_map_'+data_name+'_uncertainty_GIA','VLM_map_'+data_name+'_rob_uncertainty','VLM_map_'+data_name+'_rob_uncertainty_GIA']:
            ds_grouped['combined_unc_'+name] = np.sqrt((ds_grouped['ASL_uncertainty']/2)**2 + ds_grouped[name]**2 ) 


In [ ]:
rename_map = {var: re.sub(r"[()]", "", var) for var in ds_grouped.data_vars}
ds_grouped = ds_grouped.rename(rename_map)
ds_grouped.drop('time').to_netcdf('_temp_/ds_grouped_1.nc')
